In [103]:
# #@title Install dependencies { form-width: "20%" }

# these ones should already be available
!pip install numpy
!pip install tqdm
!pip install pillow

!pip install torch
!pip install torchvision

!pip install thop
!pip install matplotlib
!pip install scikit-learn
!pip install torchinfo
!pip install onnx2pytorch
!pip install onnxscript
!pip install onnxoptimizer

# You will need these if running on a private server (no colab)
# !pip install jupyter
# !pip install ipywidgets
# !jupyter nbextension enable --py widgetsnbextension

# QuantLab
!pip install onnx
!pip install graphviz
!rm -rf quantlab; git clone https://github.com/pulp-platform/quantlab.git
!cd quantlab; rm -rf quantlib; git clone https://github.com/pulp-platform/quantlib.git; cd quantlib; git checkout bff535
!cd quantlab/quantlib; python setup.py install

Cloning into 'quantlab'...
remote: Enumerating objects: 4037, done.
remote: Counting objects: 100% (262/262), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 4037 (delta 218), reused 218 (delta 215), pack-reused 3775 (from 1)
Receiving objects: 100% (4037/4037), 4.05 MiB | 16.44 MiB/s, done.
Resolving deltas: 100% (2404/2404), done.
Cloning into 'quantlib'...
remote: Enumerating objects: 5120, done.
remote: Counting objects: 100% (707/707), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 5120 (delta 656), reused 605 (delta 605), pack-reused 4413 (from 2)
Receiving objects: 100% (5120/5120), 1.16 MiB | 5.73 MiB/s, done.
Resolving deltas: 100% (3111/3111), done.
Note: switching to 'bff535'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to ret

In [104]:
#basic
import os
from os.path import join
import numpy as np
from tqdm import tqdm
import time

#plotting
import matplotlib.pyplot as plt

#torch
import torch; print('\nPyTorch version in use:', torch.__version__, '\ncuda avail: ', torch.cuda.is_available())
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

#torchvision
import torchvision
from torchvision import transforms, datasets

# others
from copy import deepcopy
from tqdm.autonotebook import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import inspect
from collections import defaultdict
import random

# QuantLab
import quantlib
import quantlib.editing.lightweight as qlw
import quantlib.editing.fx as qfx
import quantlib.algorithms as qa
import quantlib.backends as qb
from typing import Union, Tuple, List, Dict

import onnx
from onnx2pytorch import ConvertModel
import onnxoptimizer



PyTorch version in use: 2.11.0+cu130 
cuda avail:  False


In [105]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device: %s' % device)

Device: cpu


In [106]:
def load_net_from_onnx(model_path, device):
    onnx_model = onnx.load(model_path)
    pytorch_model = ConvertModel(onnx_model)
    pytorch_model.onnx_model = onnx_model
    pytorch_model.to(device)
    pytorch_model.eval()
    return pytorch_model

def fix_linear_metadata(model: nn.Module):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            w_out, w_in = module.weight.shape

            if module.out_features != w_out or module.in_features != w_in:
                print(
                    f"Fixing {name}: "
                    f"(in={module.in_features}, out={module.out_features}) "
                    f"-> (in={w_in}, out={w_out})"
                )
                module.in_features = w_in
                module.out_features = w_out

In [107]:
path_onnx_model = "../dory/dory_examples/examples/Custom/ES-DNN1/ES-DNN1.onnx"
path_onnx_model_DORY = "../dory/dory_examples/examples/Custom/ES-DNN1/ES-DNN1_DORY.onnx"
path_onnx_model_no_identity = "../dory/dory_examples/examples/Custom/ES-DNN1/ES-DNN1_DORY_no_identity.onnx"
path_onnx_model_numeric = "../dory/dory_examples/examples/Custom/ES-DNN1/ES-DNN1_DORY_numeric.onnx"

In [108]:
# Model: ES-DNN2
# class PlainMLP(nn.Module):
#     def __init__(self, with_softmax=True):
#         super().__init__()
#         layers = [
#             nn.Linear(16, 1024),
#             nn.ReLU(),
#             nn.Linear(1024, 704),
#             nn.ReLU(),
#             nn.Linear(704, 288),
#             nn.ReLU(),
#             nn.Linear(288, 64),
#             nn.ReLU(),
#             nn.Linear(64, 10),
#         ]
#         if with_softmax:
#             layers.append(nn.Softmax(dim=1))
#         self.net = nn.Sequential(*layers)

#     def forward(self, x):
#         return self.net(x)
    
# Model: ES-DNN1
class PlainMLP(nn.Module):
    def __init__(self, with_softmax=False):
        super().__init__()
        layers = [
            nn.Linear(16, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        ]
        if with_softmax:
            layers.append(nn.Softmax(dim=1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [109]:
def rebuild_plain_mlp_from_converted_model(src_model: nn.Module,
                                           device,
                                           with_softmax=False) -> nn.Module:
    dst_model = PlainMLP(with_softmax=with_softmax).to(device)
    dst_model.eval()

    src_linears = [m for m in src_model.modules() if isinstance(m, nn.Linear)]
    dst_linears = [m for m in dst_model.modules() if isinstance(m, nn.Linear)]

    for i, (src, dst) in enumerate(zip(src_linears, dst_linears), start=1):
        assert src.weight.shape == dst.weight.shape, \
            f"Layer {i}: weight mismatch {src.weight.shape} vs {dst.weight.shape}"

        dst.weight.data.copy_(src.weight.data)
        if src.bias is not None and dst.bias is not None:
            dst.bias.data.copy_(src.bias.data)

    return dst_model

In [110]:
# onnx_model = onnx.load("../../dory/dory/dory_examples/examples/Custom/ES-DNN1/ES-DNN1.onnx")
onnx_model = onnx.load(path_onnx_model)
# onnx_model = onnx.load("../../dory/dory/dory_examples/examples/Custom/NoFS-DNN1/NoFS-DNN1.onnx")
# onnx_model = onnx.load("../../dory/dory/dory_examples/examples/Custom/NoFS-DNN2/NoFS-DNN2.onnx")
print(onnx.helper.printable_graph(onnx_model.graph))

graph tf2onnx (
  %args_0:0[FLOAT, unk__6x16]
) initializers (
  %sequential_1/dense_3_1/Cast/ReadVariableOp:0[FLOAT, 32x10]
  %sequential_1/dense_3_1/BiasAdd/ReadVariableOp:0[FLOAT, 10]
  %sequential_1/dense_2_1/Cast/ReadVariableOp:0[FLOAT, 64x32]
  %sequential_1/dense_2_1/BiasAdd/ReadVariableOp:0[FLOAT, 32]
  %sequential_1/dense_1_2/Cast/ReadVariableOp:0[FLOAT, 128x64]
  %sequential_1/dense_1_2/BiasAdd/ReadVariableOp:0[FLOAT, 64]
  %sequential_1/dense_1/Cast/ReadVariableOp:0[FLOAT, 16x128]
  %sequential_1/dense_1/BiasAdd/ReadVariableOp:0[FLOAT, 128]
) {
  %sequential_1/dense_1/MatMul:0 = MatMul(%args_0:0, %sequential_1/dense_1/Cast/ReadVariableOp:0)
  %sequential_1/dense_1/BiasAdd:0 = Add(%sequential_1/dense_1/MatMul:0, %sequential_1/dense_1/BiasAdd/ReadVariableOp:0)
  %sequential_1/dense_1/Relu:0 = Relu(%sequential_1/dense_1/BiasAdd:0)
  %sequential_1/dense_1_2/MatMul:0 = MatMul(%sequential_1/dense_1/Relu:0, %sequential_1/dense_1_2/Cast/ReadVariableOp:0)
  %sequential_1/dense_1_2/Bi

/tmp/ipykernel_562694/973764577.py:5: DeprecationWarning: Deprecated since 1.19. Consider using onnx.printer.to_text() instead.
  print(onnx.helper.printable_graph(onnx_model.graph))


In [111]:
model = load_net_from_onnx(path_onnx_model, device)

fix_linear_metadata(model)

for name, module in model.named_modules():
    if name == "":
        continue
    print(f"{name:40s} {module.__class__.__name__}")

MatMul_sequential_1/dense_1/BiasAdd:0    Linear
Relu_sequential_1/dense_1/Relu:0         ReLU
MatMul_sequential_1/dense_1_2/BiasAdd:0  Linear
Relu_sequential_1/dense_1_2/Relu:0       ReLU
MatMul_sequential_1/dense_2_1/BiasAdd:0  Linear
Relu_sequential_1/dense_2_1/Relu:0       ReLU
MatMul_sequential_1/dense_3_1/BiasAdd:0  Linear
Softmax_Identity:0                       Softmax


In [112]:
plain_model = rebuild_plain_mlp_from_converted_model(model, device, with_softmax=False)
plain_model.eval()

for name, module in plain_model.named_modules():
    if name == "":
        continue
    print(f"{name:20s} {module.__class__.__name__}")

net                  Sequential
net.0                Linear
net.1                ReLU
net.2                Linear
net.3                ReLU
net.4                Linear
net.5                ReLU
net.6                Linear


Note: comment out the next layer if you want the original model

In [113]:
model = plain_model

In [114]:
def all_pact_f2f_recipe(network: nn.Module, name2config: Dict[str, Dict]) -> nn.Module:

    lwg = qlw.LightweightGraph(network)
    name2type = {n.name: n.module.__class__.__name__ for n in lwg.nodes_list}

    assert set(name2config.keys()).issubset(set(name2type.keys()))

    type2rule = {
        'Linear': qlw.rules.pact.ReplaceConvLinearPACTRule,
        'ReLU':   qlw.rules.pact.ReplaceActPACTRule,
    }

    rhos = [
        type2rule[name2type[n]](qlw.rules.NameFilter(n), **name2config[n])
        for n in name2config.keys()
    ]

    lwe = qlw.LightweightEditor(lwg)
    lwe.startup()
    for rho in rhos:
        lwe.set_lwr(rho)
        lwe.apply()
    lwe.shutdown()

    return lwe.graph.net

In [115]:
def all_pact_create_configs_int8(network: nn.Module, patches: Dict[str, Dict] = None) -> Dict[str, Dict]:

    if patches is None:
        patches = {}

    lwg = qlw.LightweightGraph(network)

    linear_nodes = {n.name for n in lwg.nodes_list if n.module.__class__.__name__ == 'Linear'}
    relu_nodes   = {n.name for n in lwg.nodes_list if n.module.__class__.__name__ == 'ReLU'}

    assert set(patches.keys()).issubset(linear_nodes | relu_nodes)

    linear_default = {
        'quantize':   'per_layer',
        'init_clip':  'sawb_asymm',
        'learn_clip': False,
        'symm_wts':   True,
        'tqt':        False,
        'n_levels':   256,
    }

    relu_default = {
        'init_clip':  'std',
        'learn_clip': True,
        'nb_std':     3,
        'rounding':   False,
        'tqt':        False,
        'n_levels':   256,
    }

    linear_config = defaultdict(lambda: linear_default.copy())
    for n in linear_nodes:
        linear_config[n].update(patches[n] if n in patches else {})

    relu_config = defaultdict(lambda: relu_default.copy())
    for n in relu_nodes:
        relu_config[n].update(patches[n] if n in patches else {})

    config = {**linear_config, **relu_config}
    return config

In [116]:
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        print(
            f"{name}\n"
            f"  in_features={module.in_features}, out_features={module.out_features}\n"
            f"  weight.shape={tuple(module.weight.shape)}\n"
        )

net.0
  in_features=16, out_features=128
  weight.shape=(128, 16)

net.2
  in_features=128, out_features=64
  weight.shape=(64, 128)

net.4
  in_features=64, out_features=32
  weight.shape=(32, 64)

net.6
  in_features=32, out_features=10
  weight.shape=(10, 32)



In [117]:
patches = {}

name2config = all_pact_create_configs_int8(deepcopy(model), patches)
pact_model = all_pact_f2f_recipe(deepcopy(model), name2config)
pact_model = pact_model.to(device)
pact_model.eval()

print(pact_model)

PlainMLP(
  (net): Sequential(
    (0): PACTLinear(in_features=16, out_features=128, bias=True, n_levels=256, quantize='per_layer', init_clip='sawb_asymm', learn_clip=False, symm_wts=True, nb_std=3, tqt=False, tqt_beta=0.90, tqt_clip_grad=True)
    (1): PACTUnsignedAct(n_levels=256, init_clip='std', learn_clip=True, act_kind='relu', leaky=0.1, nb_std=3, tqt=False, tqt_beta=0.90, tqt_clip_grad=True)
    (2): PACTLinear(in_features=128, out_features=64, bias=True, n_levels=256, quantize='per_layer', init_clip='sawb_asymm', learn_clip=False, symm_wts=True, nb_std=3, tqt=False, tqt_beta=0.90, tqt_clip_grad=True)
    (3): PACTUnsignedAct(n_levels=256, init_clip='std', learn_clip=True, act_kind='relu', leaky=0.1, nb_std=3, tqt=False, tqt_beta=0.90, tqt_clip_grad=True)
    (4): PACTLinear(in_features=64, out_features=32, bias=True, n_levels=256, quantize='per_layer', init_clip='sawb_asymm', learn_clip=False, symm_wts=True, nb_std=3, tqt=False, tqt_beta=0.90, tqt_clip_grad=True)
    (5): PACTU

In [118]:
for name, module in pact_model.named_modules():
    if name == "":
        continue
    print(f"{name:40s} {module.__class__.__name__}")

net                                      Sequential
net.0                                    PACTLinear
net.1                                    PACTUnsignedAct
net.2                                    PACTLinear
net.3                                    PACTUnsignedAct
net.4                                    PACTLinear
net.5                                    PACTUnsignedAct
net.6                                    PACTLinear


In [119]:
for i in range(5):
    x = torch.randn(1, 16).to(device)

    with torch.no_grad():
        y_float = model(x)
        y_fakeq = pact_model(x)

    print(f"Test {i+1}")
    print("  float class :", y_float.argmax(dim=1).item())
    print("  fakeq class :", y_fakeq.argmax(dim=1).item())
    print("  max abs diff:", (y_float - y_fakeq).abs().max().item())
    print()

Test 1
  float class : 6
  fakeq class : 6
  max abs diff: 0.0

Test 2
  float class : 6
  fakeq class : 6
  max abs diff: 0.0

Test 3
  float class : 7
  fakeq class : 7
  max abs diff: 0.0

Test 4
  float class : 3
  fakeq class : 3
  max abs diff: 0.0

Test 5
  float class : 2
  fakeq class : 2
  max abs diff: 0.0



In [120]:
def get_input_range(data_loader: torch.utils.data.DataLoader) -> Tuple[float, float]:
    min_ = 0.0
    max_ = 0.0

    for x, _ in data_loader:
        min_ = min(min_, x.min().item())
        max_ = max(max_, x.max().item())

    return min_, max_

In [121]:
def add_quantisation_transform(data_loader: torch.utils.data.DataLoader,
                               n_levels: int,
                               min_: float,
                               max_: float) -> float:

    quantiser = qa.pact.PACTAsymmetricAct(
        n_levels=n_levels,
        symm=True,
        learn_clip=False,
        init_clip='max',
        act_kind='identity'
    )

    clip_lo, clip_hi = qa.pact.util.almost_symm_quant(
        torch.Tensor([max(abs(min_), abs(max_))]),
        n_levels
    )

    quantiser.clip_lo.data = clip_lo
    quantiser.clip_hi.data = clip_hi
    quantiser.started |= True

    transform_list = []

    if data_loader.dataset.transform is not None:
        transform_list.append(data_loader.dataset.transform)

    transform_list.append(quantiser)
    transform_list.append(transforms.Lambda(lambda x: x / quantiser.get_eps()))

    data_loader.dataset.transform = transforms.Compose(transform_list)

    return quantiser.get_eps()

In [122]:
def f2t_convert(dataloader: torch.utils.data.DataLoader,
                input_eps: float,
                network: nn.Module) -> nn.Module:

    network.eval()
    network = network.to(device=torch.device('cpu'))

    x, _ = dataloader.dataset.__getitem__(0)
    x = x.unsqueeze(0).to(device=torch.device('cpu'))

    fake2true_converter = qfx.passes.pact.IntegerizePACTNetPass(
        shape_in=x.shape,
        eps_in=input_eps,
        D=2**19
    )

    return fake2true_converter(network)

In [123]:
class VectorDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]

        if self.transform is not None:
            x = self.transform(x)

        return x, y

In [124]:
N = 128
X_dummy = torch.randn(N, 16)
y_dummy = torch.zeros(N, dtype=torch.long)

dummy_dataset = VectorDataset(X_dummy, y_dummy, transform=None)
dummy_loader = DataLoader(dummy_dataset, batch_size=32, shuffle=False)

In [125]:
n_input_levels = 2**8

min_, max_ = get_input_range(dummy_loader)
print("Input range:", min_, max_)

dummy_loader_true_quant = deepcopy(dummy_loader)
input_eps = add_quantisation_transform(dummy_loader_true_quant, n_input_levels, min_, max_)

print("input_eps:", input_eps)

Input range: -3.5829293727874756 3.587749719619751
input_eps: tensor([0.0280])


In [126]:
tq_pact_model = f2t_convert(dummy_loader_true_quant, input_eps, deepcopy(pact_model))
print(tq_pact_model)

key _OUTPUT_output not found in _EPS_CONVERSIONS!
Using identity epsilon propagation on node with op output, target output!
PlainMLP(
  (net): Module(
    (_QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_0): Linear(in_features=16, out_features=128, bias=True)
    (_QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_1): Linear(in_features=128, out_features=64, bias=True)
    (_QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_2): Linear(in_features=64, out_features=32, bias=True)
    (_QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_3): Linear(in_features=32, out_features=10, bias=True)
    (_QL_REPLACED__INTEGERIZE_UNSIGNED_ACT_PASS_0): RequantShift()
    (_QL_REPLACED__INTEGERIZE_UNSIGNED_ACT_PASS_1): RequantShift()
    (_QL_REPLACED__INTEGERIZE_UNSIGNED_ACT_PASS_2): RequantShift()
  )
)



def forward(self, x):
    net__ql_replaced__integerize_pact_lin_pass_0 = self.net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_0(x);  x = None
    net__ql_replaced__integerize_unsigned_act_pass_0 = self.net._QL_REPLACED__INTEGERIZE_UNSIGNED_ACT_PA

In [127]:
x_export = dummy_loader_true_quant.dataset[0][0].unsqueeze(0).cpu()
print(x_export.shape)
print(x_export.dtype)
print(x_export)

torch.Size([1, 16])
torch.float32
tensor([[ 39.0000,  13.0000, -51.0000,   8.0000,  31.0000,  47.0000, -30.0000,
         -32.0000,  69.0000, -30.0000, -10.0000, -45.0000, -25.0000, -22.0000,
          38.0000,  42.0000]])


In [128]:
tq_pact_model.eval()

torch.onnx.export(
    tq_pact_model,
    x_export,
    path_onnx_model_DORY,
    input_names=["input"],
    output_names=["output"],
    opset_version=9,
    do_constant_folding=True
)

W0413 15:55:11.988000 562694 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 9 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `PlainMLP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PlainMLP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 9).
Failed to convert the model to the target version 9 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/pierre/Documents/thesis/projects-pulp/dory/envDory/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pierre/Documents/thesis/projects-pulp/dory/envDory/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/home/p

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 3 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,16]>
            ),
            outputs=(
                %"output"<FLOAT,[1,10]>
            ),
            initializers=(
                %"net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_0.bias"<FLOAT,[128]>{TorchTensor(...)},
                %"net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_1.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_2.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_3.weight"<FLOAT,[10,32]>{TorchTensor(...)},
                %"net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_3.bias"<FLOAT,[10]>{TorchTensor<FLOAT,[10]>(Parameter containing

In [129]:
onnx_model_int = onnx.load(path_onnx_model_DORY)
onnx.checker.check_model(onnx_model_int)
print("ONNX export OK")

for node in onnx_model_int.graph.node:
    print(node.op_type)

ONNX export OK
Gemm
Mul
Div
Floor
Clip
Gemm
Mul
Div
Floor
Clip
Gemm
Mul
Div
Floor
Clip
Gemm


In [130]:
src = path_onnx_model_DORY
dst = path_onnx_model_no_identity

model = onnx.load(src)

passes = [
    "eliminate_identity",
]

model_opt = onnxoptimizer.optimize(model, passes)
onnx.checker.check_model(model_opt)
onnx.save(model_opt, dst)

print("Saved:", dst)
print("Nodes after cleanup:")
for node in model_opt.graph.node:
    print(node.op_type)

Saved: ../dory/dory_examples/examples/Custom/ES-DNN1/ES-DNN1_DORY_no_identity.onnx
Nodes after cleanup:
Gemm
Mul
Div
Floor
Clip
Gemm
Mul
Div
Floor
Clip
Gemm
Mul
Div
Floor
Clip
Gemm


In [131]:
def collect_tensor_names(model):
    names = []

    for vi in model.graph.input:
        if vi.name:
            names.append(vi.name)

    for vi in model.graph.output:
        if vi.name:
            names.append(vi.name)

    for vi in model.graph.value_info:
        if vi.name:
            names.append(vi.name)

    for init in model.graph.initializer:
        if init.name:
            names.append(init.name)

    for node in model.graph.node:
        for x in node.input:
            if x:
                names.append(x)
        for x in node.output:
            if x:
                names.append(x)

    # preserve first-seen order
    seen = set()
    ordered = []
    for n in names:
        if n not in seen:
            seen.add(n)
            ordered.append(n)

    return ordered


def rename_all_tensors_to_numeric(src_path, dst_path):
    model = onnx.load(src_path)

    tensor_names = collect_tensor_names(model)
    name_map = {old: str(i) for i, old in enumerate(tensor_names)}

    # graph inputs
    for vi in model.graph.input:
        if vi.name in name_map:
            vi.name = name_map[vi.name]

    # graph outputs
    for vi in model.graph.output:
        if vi.name in name_map:
            vi.name = name_map[vi.name]

    # intermediate value_info
    for vi in model.graph.value_info:
        if vi.name in name_map:
            vi.name = name_map[vi.name]

    # initializers
    for init in model.graph.initializer:
        if init.name in name_map:
            init.name = name_map[init.name]

    # nodes
    for node in model.graph.node:
        for i, x in enumerate(node.input):
            if x in name_map:
                node.input[i] = name_map[x]
        for i, x in enumerate(node.output):
            if x in name_map:
                node.output[i] = name_map[x]

    onnx.checker.check_model(model)
    onnx.save(model, dst_path)

    print(f"Saved: {dst_path}")
    print("Example mapping:")
    for k in list(name_map.keys())[:20]:
        print(f"{k} -> {name_map[k]}")


rename_all_tensors_to_numeric(
    path_onnx_model_no_identity,
    path_onnx_model_numeric
)

Saved: ../dory/dory_examples/examples/Custom/ES-DNN1/ES-DNN1_DORY_numeric.onnx
Example mapping:
input -> 0
output -> 1
linear -> 2
mul -> 3
div -> 4
floor -> 5
clamp -> 6
linear_1 -> 7
mul_1 -> 8
div_1 -> 9
floor_1 -> 10
clamp_1 -> 11
linear_2 -> 12
mul_2 -> 13
div_2 -> 14
floor_2 -> 15
clamp_2 -> 16
net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_0.bias -> 17
net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_1.bias -> 18
net._QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_2.bias -> 19


In [132]:
m = onnx.load(path_onnx_model_numeric)
onnx.checker.check_model(m)

for node in m.graph.node[:10]:
    print(node.op_type, list(node.input), list(node.output))

Gemm ['0', '24', '17'] ['2']
Mul ['2', '22'] ['3']
Div ['3', '27'] ['4']
Floor ['4'] ['5']
Clip ['5', '28', '29'] ['6']
Gemm ['6', '25', '18'] ['7']
Mul ['7', '23'] ['8']
Div ['8', '27'] ['9']
Floor ['9'] ['10']
Clip ['10', '28', '29'] ['11']


In [133]:
# Example: one quantized input sample
x_q, _ = dummy_loader_true_quant.dataset[0]
x_q = x_q.unsqueeze(0).cpu()

with torch.no_grad():
    y_q = tq_pact_model(x_q)

print("input shape:", x_q.shape)
print("output shape:", y_q.shape)
print("input:", x_q)
print("output:", y_q)

input shape: torch.Size([1, 16])
output shape: torch.Size([1, 10])
input: tensor([[ 39.0000,  13.0000, -51.0000,   8.0000,  31.0000,  47.0000, -30.0000,
         -32.0000,  69.0000, -30.0000, -10.0000, -45.0000, -25.0000, -22.0000,
          38.0000,  42.0000]])
output: tensor([[-161687., -142028.,  -78133.,  -56512.,  -44067., -140907.,   33260.,
         -135828., -144230.,  -51810.]])


In [134]:
# ------------------------------------------------------------------
# 1) Collect the 4 Linear layers in actual forward order
# ------------------------------------------------------------------
executed_modules = []

def make_hook(name):
    def hook(module, inputs, output):
        executed_modules.append((name, module))
    return hook

hooks = []
for name, module in tq_pact_model.net.named_modules():
    if name == "":
        continue
    hooks.append(module.register_forward_hook(make_hook(name)))

x_q, _ = dummy_loader_true_quant.dataset[0]
x = x_q.unsqueeze(0).cpu()

with torch.no_grad():
    _ = tq_pact_model(x)

for h in hooks:
    h.remove()

linear_layers = []
for name, module in executed_modules:
    if module.__class__.__name__ == "Linear":
        linear_layers.append((name, module))

print("Linear layers in execution order:")
for i, (name, module) in enumerate(linear_layers):
    print(i, name, tuple(module.weight.shape))

# assert len(linear_layers) == 4, f"Expected 4 Linear layers, got {len(linear_layers)}"


Linear layers in execution order:
0 _QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_0 (128, 16)
1 _QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_1 (64, 128)
2 _QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_2 (32, 64)
3 _QL_REPLACED__INTEGERIZE_PACT_LIN_PASS_3 (10, 32)


In [140]:
# ------------------------------------------------------------------
# 2) Exact PULP arithmetic
# ------------------------------------------------------------------
def clip8(x: int) -> int:
    return max(0, min(255, int(x)))

def pulp_nn_quant_u8(phi: int, m: int, d: int) -> int:
    # Emulate C int32_t overflow in (m * phi)
    phi_i32 = np.int32(phi)
    m_i16 = np.int16(m)

    prod_i64 = np.int64(m_i16) * np.int64(phi_i32)
    prod_i32 = np.int32(prod_i64)   # wrap to int32 like C on target
    x_i32 = np.int32(prod_i32 >> np.int8(d))  # arithmetic right shift

    return clip8(int(x_i32))
        
def fc_i8_u8_i8_onnx_requant(
    x_i8: np.ndarray,
    w_i8: np.ndarray,
    b_i32: np.ndarray,
    out_mult: int,
    out_shift: int,
    post_div_add: float = 0.5,
) -> np.ndarray:
    x_i8 = np.asarray(x_i8, dtype=np.int8).reshape(-1)
    w_i8 = np.asarray(w_i8, dtype=np.int8)
    b_i32 = np.asarray(b_i32, dtype=np.int32).reshape(-1)

    y = np.zeros((w_i8.shape[0],), dtype=np.uint8)
    div_val = float(2 ** int(out_shift))

    for i in range(w_i8.shape[0]):
        phi = int(b_i32[i]) + int(np.dot(w_i8[i].astype(np.int64), x_i8.astype(np.int64)))

        prod_i64 = np.int64(np.int16(out_mult)) * np.int64(np.int32(phi))
        prod_i32 = np.int32(prod_i64)

        z = float(prod_i32) / div_val
        z = z + float(post_div_add)
        z = np.floor(z)

        if z < 0:
            z = 0
        elif z > 255:
            z = 255

        y[i] = np.uint8(z)

    return y

def fc_u8_u8_i8(x_u8: np.ndarray,
                w_i8: np.ndarray,
                b_i32: np.ndarray,
                out_mult: int,
                out_shift: int) -> np.ndarray:
    x_u8 = np.asarray(x_u8, dtype=np.uint8).reshape(-1)
    y = np.zeros(w_i8.shape[0], dtype=np.uint8)

    for i in range(w_i8.shape[0]):
        phi = int(b_i32[i]) + int(np.dot(w_i8[i].astype(np.int64), x_u8.astype(np.int64)))
        y[i] = pulp_nn_quant_u8(phi, out_mult, out_shift)

    return y

def fc_i8_u8_i8(
    x_i8: np.ndarray,
    w_i8: np.ndarray,
    b_i32: np.ndarray,
    out_mult: int,
    out_shift: int,
) -> np.ndarray:
    x_i8 = np.asarray(x_i8, dtype=np.int8).reshape(-1)
    w_i8 = np.asarray(w_i8, dtype=np.int8)
    b_i32 = np.asarray(b_i32, dtype=np.int32).reshape(-1)

    y = np.zeros(w_i8.shape[0], dtype=np.uint8)

    for i in range(w_i8.shape[0]):
        phi = int(b_i32[i]) + int(np.dot(w_i8[i].astype(np.int64), x_i8.astype(np.int64)))
        y[i] = pulp_nn_quant_u8(phi, out_mult, out_shift)

    return y
    
def fc_u8_i32_i8(x_u8: np.ndarray,
                 w_i8: np.ndarray,
                 b_i32: np.ndarray) -> np.ndarray:
    x_u8 = np.asarray(x_u8, dtype=np.uint8).reshape(-1)
    y = np.zeros(w_i8.shape[0], dtype=np.int32)

    for i in range(w_i8.shape[0]):
        phi = int(b_i32[i]) + int(np.dot(w_i8[i].astype(np.int64), x_u8.astype(np.int64)))
        y[i] = np.int32(phi)

    return y

def extract_linear_params(linear_module):
    # Torch Linear uses weight shape (out_features, in_features)
    w = linear_module.weight.detach().cpu().numpy()
    b = linear_module.bias.detach().cpu().numpy()

    # Integerized models often still store integer values in float tensors.
    w_i8 = np.rint(w).astype(np.int8)
    b_i32 = np.rint(b).astype(np.int32)

    return w_i8, b_i32


In [144]:
# ------------------------------------------------------------------
# 3) Requant parameters
# ------------------------------------------------------------------
out_mult_vector  = [31126, 4112, 4112]   # len = n_layers - 1
out_shift_vector = [19, 19, 19]

# ------------------------------------------------------------------
# 4) Extract all Linear layers dynamically
# ------------------------------------------------------------------
linear_modules = [m for _, m in linear_layers]  # assuming your structure
n_layers = len(linear_modules)

weights = []
biases = []

for i, layer in enumerate(linear_modules):
    W, B = extract_linear_params(layer)
    weights.append(W)
    biases.append(B)
    print(f"Layer {i}: W{W.shape}, B{B.shape}")

# ------------------------------------------------------------------
# 5) Integer forward (dynamic)
# ------------------------------------------------------------------
x = x_q.detach().cpu().numpy().astype(np.uint8).reshape(-1)

outputs = [x]  # store all intermediate outputs

for i in range(n_layers):
    if i < n_layers - 1:
        # Intermediate layers (with requant)
        y = fc_u8_u8_i8(
            outputs[-1],
            weights[i],
            biases[i],
            out_mult_vector[i],
            out_shift_vector[i],
        )
    else:
        # Last layer (no requant)
        y = fc_u8_i32_i8(
            outputs[-1],
            weights[i],
            biases[i],
        )
    
    outputs.append(y)

# ------------------------------------------------------------------
# 6) Print checksums
# ------------------------------------------------------------------
print("\nChecksums:")
for i, out in enumerate(outputs):
    if i == 0:
        print(f"input      {int(out.sum())}")
    else:
        print(f"out_layer{i-1} {int(out.sum())}")

# ------------------------------------------------------------------
# 7) Save txt files
# ------------------------------------------------------------------
out_dir = "dory_reference_outputs"
os.makedirs(out_dir, exist_ok=True)

# Save input
np.savetxt(os.path.join(out_dir, "input.txt"),
           outputs[0].reshape(-1, 1), fmt="%d", delimiter=",")

# Save all layers
for i in range(1, len(outputs)):
    np.savetxt(
        os.path.join(out_dir, f"out_layer{i-1}.txt"),
        outputs[i].reshape(-1, 1),
        fmt="%d",
        delimiter=","
    )

print(f"\nSaved files to: {out_dir}")

Layer 0: W(128, 16), B(128,)
Layer 1: W(64, 128), B(64,)
Layer 2: W(32, 64), B(32,)
Layer 3: W(10, 32), B(10,)

Checksums:
input      2093
out_layer0 14964
out_layer1 5355
out_layer2 1740
out_layer3 -334289

Saved files to: dory_reference_outputs


In [137]:
ref = np.loadtxt(os.path.join("dory_reference_outputs/out_layer0.txt"), delimiter=",", dtype=np.int64, usecols=[0])
dory_ref = np.loadtxt(os.path.join("../../dory/dory/dory_examples/examples/Custom/out_layer0.txt"), delimiter=",", dtype=np.int64, usecols=[0])

print("sum(ref)     =", int(ref.sum()))
print("sum(doryref) =", int(dory_ref.sum()))
print("equal:", np.array_equal(ref, dory_ref))

diff_idx = np.where(ref != dory_ref)[0]
print("num diffs:", len(diff_idx))
print("first diffs:", diff_idx[:20].tolist())
for i in diff_idx[:10]:
    print(i, int(ref[i]), int(dory_ref[i]))

sum(ref)     = 14964
sum(doryref) = 16371
equal: False
num diffs: 71
first diffs: [1, 2, 6, 7, 8, 11, 12, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 29, 31, 34]
1 95 255
2 0 255
6 0 255
7 0 255
8 255 0
11 0 255
12 0 255
16 255 0
17 0 255
18 255 0


In [138]:
print("W0[0] =", W0[0].tolist())
print("W0[1] =", W0[1].tolist())
print("B0[:8] =", B0[:8].tolist())

W0[0] = [-99, 25, -127, 21, 20, 9, -116, -127, 127, -127, 127, -117, 24, 127, -62, 96]
W0[1] = [24, -127, -127, -127, 0, 40, 43, 127, -127, 36, -127, -6, 6, 127, -127, -51]
B0[:8] = [1387, -2082, -291, 390, -197, 1448, -2405, -1509]
